# Pilot sample after exclusions (v4 → 998 accounts)

**Pool rule 8 (2026-10-07): exclude Reddit internal test accounts** — usernames starting `lss_mobile`, `lss_web`,
`lss_bvt` or `lssqa` (case-insensitive). These accounts post almost only in Reddit's internal test subreddits (`lss…`,
RedditMobTest) and were accused on TFA as "a network of bots and subs". `example_viral` does not match and is kept (check it
before the larger draw).

The pilot drops its two matching accounts **without replacement** (replacements would bring in unclassified subreddits).
Inputs: `sample_1000_v4_output/`. Outputs (never overwritten): `sample_998_v4_excl_output/` — the 998 accounts, their
posts/comments, the pool after rule 8, and the excluded accounts with the reason.

In [ ]:
# Libraries, folders, and the rule-8 pattern: Reddit internal test accounts
# (usernames starting lss_mobile, lss_web, lss_bvt or lssqa, any case).
import pandas as pd
import numpy as np
import os
import re
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

IN_DIR = 'sample_1000_v4_output'
OUT_DIR = 'sample_998_v4_excl_output'
TEST_ACCOUNT = re.compile(r'^(lss_mobile|lss_web|lss_bvt|lssqa)', re.I)
os.makedirs(OUT_DIR, exist_ok=True)

# Write a parquet file only if it does not exist yet (never overwrite a saved sample).
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

In [ ]:
# Apply rule 8 to the eligible pool. Usernames are joined on only to test the pattern,
# then dropped again before saving.
pool = pd.read_parquet(os.path.join(IN_DIR, 'eligible_pool.parquet'))
names = pd.read_parquet(os.path.join('..', 'p2_data_cleaning', 'data', 'accused_accounts_not_deleted.parquet'),
                        columns=['username_key', 'username'])
pool = pool.merge(names, on='username_key', how='left', validate='one_to_one')
pool['internal_test_account'] = pool['username'].str.match(TEST_ACCOUNT)
print('pool accounts matching rule 8:', int(pool['internal_test_account'].sum()))
print(pool.loc[pool['internal_test_account'], ['username', 'account_status']].to_string(index=False))
pool_after = pool[~pool['internal_test_account']].drop(columns=['username', 'internal_test_account']).reset_index(drop=True)
print('pool after rule 8:', len(pool_after))

In [ ]:
# Apply rule 8 to the 1,000-account v4 sample and record which accounts were dropped and why.
accounts = pd.read_parquet(os.path.join(IN_DIR, 'sample_accounts.parquet'))
drop = accounts['username'].str.match(TEST_ACCOUNT)
excluded = accounts.loc[drop, ['username_key', 'username', 'account_status']].assign(reason='rule 8: Reddit internal test account')
print(excluded.to_string(index=False))
# Keep the remaining 998 accounts and only their posts and comments.
keep = set(accounts.loc[~drop, 'username_key'])
sample_accounts = accounts[~drop].reset_index(drop=True)
sample_posts = pd.read_parquet(os.path.join(IN_DIR, 'sample_posts.parquet')).loc[lambda d: d['outer_key'].isin(keep)]
sample_comments = pd.read_parquet(os.path.join(IN_DIR, 'sample_comments.parquet')).loc[lambda d: d['outer_key'].isin(keep)]
# The rule removes exactly 2 accounts, leaving 998.
assert len(sample_accounts) == 998
print(sample_accounts['account_status'].value_counts().to_string())
print(f'posts {len(sample_posts):,}  comments {len(sample_comments):,}')

In [ ]:
# Save the pool and the sample after rule 8, plus the list of excluded accounts.
save_new(pool_after, 'eligible_pool.parquet')
save_new(sample_accounts, 'sample_accounts.parquet')
save_new(sample_posts.reset_index(drop=True), 'sample_posts.parquet')
save_new(sample_comments.reset_index(drop=True), 'sample_comments.parquet')
save_new(excluded.reset_index(drop=True), 'excluded_accounts.parquet')